In [14]:
import pandas as pd
import numpy as np

# 1. Carregar dataset de votações da AGNU (padrão Erik Voeten / UNGA Data)
df_unga = pd.read_csv('/content/UNVotes.csv')


In [15]:
# Inspeção dos objetos (Series e DataFrame)
print("Formato da Matriz Internacional:", df_unga.shape)
print(df_unga.info())

Formato da Matriz Internacional: (751418, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 751418 entries, 0 to 751417
Data columns (total 26 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   rcid           751418 non-null  int64  
 1   ccode          751418 non-null  int64  
 2   member         511908 non-null  float64
 3   vote           751418 non-null  int64  
 4   Country        751418 non-null  object 
 5   Countryname    751418 non-null  object 
 6   year           751418 non-null  int64  
 7   session        751418 non-null  int64  
 8   abstain        751418 non-null  int64  
 9   yes            751418 non-null  int64  
 10  no             751418 non-null  int64  
 11  importantvote  751418 non-null  int64  
 12  date           751418 non-null  object 
 13  unres          721671 non-null  object 
 14  amend          560268 non-null  float64
 15  para           560268 non-null  float64
 16  short          751418 non-nu

In [16]:
# 2. Operações de String vetorizadas (.str): Filtrando resoluções sobre Desarmamento e Sanções
nuclear_res = df_unga[df_unga['descr'].str.contains('nuclear|disarmament', case=False, na=False)]

In [24]:
nuclear_res['descr'].iloc[0]

'TO ADOPT AD HOC POLITICAL COMMITTEE DRAFT RESOLUTION (A/2762) DECLARING THAT THE FOREIGN ARMED FORCES STILL IN BURMESE TERRITORY SHOULD SUBMIT TO DISARMAMENT AND INTERNMENT.'

In [28]:
# 3. Seleção via .loc e .iloc: Filtrando Brasil (BRA) e EUA (USA) no pós-Guerra Fria
bra_usa_nuclear = nuclear_res.loc[
    (nuclear_res['year'] >= 1991) & (nuclear_res['Country'].isin(['BRA', 'USA'])),
    ['year', 'rcid', 'Country', 'vote', 'descr']
]

In [30]:
# Mapeamento clássico em RI: 1 = A Favor, 2 = Abstenção, 3 = Contra, 9 = Ausente/Não-membro

# Remover não-membros / ausências não registradas (vote == 9 ou NaN)
df_clean = df_unga.dropna(subset=['vote']).copy()
df_clean = df_clean[df_clean['vote'] != 9]

# Recodificar escala de concordância diplomática (Signorino & Ritter, 1999):
# A Favor = 1.0 | Abstenção = 0.5 | Contra = 0.0
vote_map = {1: 1.0, 2: 0.5, 3: 0.0}
df_clean['vote_score'] = df_clean['vote'].map(vote_map)

In [54]:
df_vote = df_clean[df_clean['year'] >= 1990]
df_vote = df_vote.groupby(['year', 'rcid'])['vote_score'].mean()

In [55]:
df_vote

year  rcid
1990  3501    0.892617
      3502    0.989796
      3503    0.844595
      3504    0.850340
      3505    0.872414
                ...   
1993  3761    0.844961
      3762    0.984177
      3763    0.957516
      3764    0.710000
      3765    0.788462
Name: vote_score, Length: 263, dtype: float64

In [56]:
# 2. Pivot Table: Transformar dado longo em Matriz Países (Linhas) vs Resoluções (Colunas)
piv_recent = df_clean[df_clean['year'] >= 1990].pivot_table(
    index='rcid',
    columns='Country',
    values='vote_score'
)

In [60]:
# 3. Índice de Concordância Diplomática com o Brasil (BRA):
# Fórmula: 1 - |Voto_Brasil - Voto_OutroPaís|
align_bra = (1 - (piv_recent.sub(piv_recent['BRA'], axis=0)).abs()).mean()
align_bra.sort_values(ascending=False)

,0
Country,
BRA,1.000000
MCO,1.000000
KHM,0.987500
SYC,0.985876
STP,0.984733
...,...
FRA,0.624506
GBR,0.568359
ERI,0.500000


In [63]:
df_clean[['Country', 'Countryname']].drop_duplicates()

,Country,Countryname
0,USA,United States of America
1,CAN,Canada
3,CUB,Cuba
4,HTI,Haiti
5,DOM,Dominican Republic
...,...,...
746093,SVK,Slovakia
746098,MKD,The former Yugoslav Republic of Macedonia
746277,MCO,Monaco
746281,AND,Andorra


In [37]:
# 2. Pivot Table: Transformar dado longo em Matriz Países (Linhas) vs Resoluções (Colunas)
piv_recent = df_clean[df_clean['year'] >= 2020].pivot_table(
    index='rcid',
    columns='Country',
    values='vote_score'
)

"A Autonomia e as Grandes Potências":

Calcule o Índice de Concordância Diplomática do Brasil (BRA) com os EUA (USA) e com a China (CHN) por década.

Qual potência esteve mais alinhada ao Brasil ao longo do tempo?

Escreva um parágrafo interpretando o resultado à luz da literatura de Política Externa Brasileira (PEB).